# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.llms.base import TextGenerator

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"

## Define Helper Functions

In [3]:
def get_feature_transforms(llm_assistant: TextGenerator, transform_code: str,
                           feature_columns: list[str],
                           feature_description: str):
    """
    Given a list of feature columns, check if the columns are transformed in the
    transform code and return the code that performs the transformation.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        transform_code (str): The code that performs the transformations.
        feature_columns (list[str]): The list of feature columns to check.
        
    Returns:
        dict: A dictionary of feature columns and the code that performs the transformation.
    """
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        performing data cleaning and preprocessing tasks."""
    transform_responses = []
    for feature_column in feature_columns:
        find_transform_prompt = f"""Given the following code:
            <Code>
            {transform_code}
            </Code>
            and the feature column:
            <Feature Column>
            {feature_column}
            </Feature Column>
            with description:
            <Feature Description>
            {feature_description}
            </Feature Description>
            determine if the column is transformed in the code. \
            If it is, return only the corresponding lines of code that perform the transformation. \
            If it is not, return "No transformation code found."
            """
        response = llm_assistant.generate([{"role": "system",
                                            "content": system_prompt},
                                           {"role": "user",
                                            "content": find_transform_prompt}])
        transform_responses.append(response)
    return transform_responses
                

In [4]:
def get_model_information(llm_assistant: TextGenerator, model_code: str):
    """
    Given modeling code, extract relevant information.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        model_code (str): The code that defines the model.
        
    Returns:
        dict: A dictionary of model information, particularly model class.
    """
    
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        choosing, identifying, and implementing different types of ML models."""
    
    find_model_prompt = f"""Given the following code:
        <Code>
        {model_code}
        </Code>
        extract relevant information about the model. The returned value should be a dictionary with the following keys:
        1. "model_library": The library or framework used (e.g., "sklearn", "statsmodels", "pytorch", "tensorflow").
        2. "model_class": The specific model class or type (e.g., "LinearRegression", "RandomForestClassifier", "LogisticRegression").
        3. "model_parameters": Any parameters or hyperparameters that are set when instantiating or configuring the model.
        4. "model_formula_fitting_code": The code that defines the model formula and/or the code that fits/trains the model.
        
        The values of the dictionary should be strings.
        """
    
    response = llm_assistant.generate([{"role": "system",
                                        "content": system_prompt},
                                       {"role": "user",
                                        "content": find_model_prompt}])
    
    return response.text[0].content
                

In [5]:
def write_final_answer_code(llm_assistant: TextGenerator, task: list[str],
                            independent_variable: dict, dependent_variable: dict,
                            model_code: str, model_output):
    """
    In its attempt to answer the underlying question, GenAI analyst writes
    two functions: one which preprocesses the data and another that performs
    some sort of analysis.
    
    The modeling function written by the GenAI analyst does not explicitly
    answer the yes/no question posed in the task. The model code that it writes
    does something very close, which is fitting a model to the data that would
    answer the question with extra interpretations.
    
    The goal of this function is to complete analysis by interpreting the model
    output. We want to accomplish the task by asking an LLM to take in the model
    output and write code that extracts the final answer from the model output.
    This code must be written in python and have a consistent function header
    to make it easy to call later in the pipeline.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        task (str): The task to be answered.
        independent_variable (dict): The independent variable to be used in the analysis.
        dependent_variable (dict): The dependent variable to be used in the analysis.
        model_code (str): The code that defines the model.
        model_output: The output of the model.
        
    Returns:
        str: A python function that reaches the conclusion of the task by
            extracting the final answer from the model output.
    """
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        drawing data-driven conclusions."""
    
    find_answer_prompt = f"""Given the following task/question:
        <Task>
        {task}
        </Task>
        
        The independent variable used in the analysis:
        <Independent Variable>
        {independent_variable}
        </Independent Variable>
        
        The dependent variable used in the analysis:
        <Dependent Variable>
        {dependent_variable}
        </Dependent Variable>
        
        The model code that was executed:
        <Model Code>
        {model_code}
        </Model Code>
        
        The output from running the modeling function (this is the raw model object, not an interpreted answer):
        <Model Output>
        {model_output}
        </Model Output>
        
        The modeling function does not explicitly answer the yes/no question posed in the task. 
        The model output is simply the raw model object returned by the model code (e.g., a fitted 
        statsmodels model, sklearn model, etc.). To answer the yes/no question, you need to:
        
        Write Python code that extracts relevant statistics from the model output object 
        (e.g., coefficients, p-values, confidence intervals, effect sizes) that relate 
        to the independent variable's effect on the dependent variable.
                
        The Python function needs to have the following function header:
        ```python
        def extract_final_answer(model_output):
            # Your code here to extract and interpret statistics from model_output
            # Return a dictionary with keys: "object", "description"
            pass
        ```
        
        The function should:
        - Take the model_output as input
        - Extract the necessary statistics from the model output object
        - Return a dictionary with:
            - "object": The actual value you would like to return (e.g. a coefficient, p-value, etc.)
            - "description": A brief explanation of the extracted statistics/return object and what it means in the context of the task
        
        Provide the complete function code that can be executed to extract the final answer.
        """
    
    response = llm_assistant.generate([{"role": "system",
                                        "content": system_prompt},
                                       {"role": "user",
                                        "content": find_answer_prompt}])
    
    return response.text[0].content
    

In [6]:
def make_conclusion(llm_assistant: TextGenerator, task: list[str],
                    independent_variable: dict, dependent_variable: dict,
                    model_code: str, interpretation_code: str,
                    interpretation_output: dict):
    """
    In its attempt to answer the underlying question, GenAI analyst writes
    three functions: one which preprocesses the data, another that performs
    some sort of analysis, and another that interprets the results.
    
    The goal of this function is to conclude the final answer by inspecting
    the interpretation of the model output. We want to accomplish the task by
    asking an LLM to take in the interpretation of the model output and
    explicitly answer the yes/no question posed in the task.
    
    Args:
        llm_assistant (TextGenerator): The LLM assistant for the evaluation.
        task (str): The task to be answered.
        independent_variable (dict): The independent variable to be used in the analysis.
        dependent_variable (dict): The dependent variable to be used in the analysis.
        model_code (str): The code that defines the model.
        interpretation_code (str): The code that interprets the model output.
        interpretation_output (dict): The output of the interpretation code.
        
    Returns:
        str: The final answer to the task. Must be either "Yes", "No", or "Not enough information".
    """
    system_prompt = """You are an AI Data Analysis Assistant who is an expert at \
        drawing data-driven conclusions from model summaries."""
    
    find_answer_prompt = f"""Given the following task/question:
        <Task>
        {task}
        </Task>
        
        The independent variable used in the analysis:
        <Independent Variable>
        {independent_variable}
        </Independent Variable>
        
        The dependent variable used in the analysis:
        <Dependent Variable>
        {dependent_variable}
        </Dependent Variable>
        
        The model code that was executed:
        <Model Code>
        {model_code}
        </Model Code>
        
        The model interpretation code that was executed:
        <Model Interpretation Code>
        {interpretation_code}
        </Model Interpretation Code>
        
        The model interpretation output:
        <Model Interpretation Output>
        {interpretation_output}
        </Model Interpretation Output>
        
        Analyze the model interpretation output and determine the final answer to the yes/no question posed in the task. 
        Return only a clear yes or no answer, with a brief justification if helpful. The final answer should be a dictionary with the following keys:
        1. "answer": The final answer to the question. Only valid options are "Yes", "No", or "Not enough information".
        2. "justification": A brief justification for the answer.
        """
    
    response = llm_assistant.generate([{"role": "system",
                                        "content": system_prompt},
                                       {"role": "user",
                                        "content": find_answer_prompt}])
    
    return response.text[0].content
    

## Extract Features **X** Used in Model

In [7]:
# create dict to store features
features = {}

In [8]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # create internal dict for analysis features
    features[i] = {}
    
    # get the features from each analysis
    # this should include the independent and control variables
    ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
    control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in ind_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    for dict_idx, var in enumerate(ind_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
        
    # save updated independent variables in features dict
    features[i]['independent_variables'] = ind_vars
    
    # tkae same approach for control variables
    for dict_idx, var in enumerate(control_vars):
        transform_responses = get_feature_transforms(llm_assistant,
                                                     transform_code,
                                                     var['columns'],
                                                     var['description'])
        control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
            response in transform_responses]
    
    # save updated control variables in features dict
    features[i]['control_variables'] = control_vars

[2025-11-12 09:02:58.06][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-12 09:02:59.28][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [9]:
# view feature dictionary to ensure correctness
features

{0: {'independent_variables': [{'description': 'Name femininity rating (continuous). This is the mean masculinity-femininity rating where larger values indicate more feminine names. Standardized for interpretability and to aid model convergence.',
    'columns': ['masfem_std'],
    'transform_code': ["df['masfem_std'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary gender label for the hurricane name (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["# Ensure gender_mf is binary numeric (0/1)\ndf['gender_mf'] = pd.to_numeric(df['gender_mf'], errors='coerce')"]}],
  'control_variables': [{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['wind'],
    'transform_code': ["df['wind'] = pd.to_numeric(df['win

## Extract Response *y* used in Model

In [10]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    
    # get the features from each analysis
    # this should include the independent and control variables
    response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

    # get any lines from the transform code that represent transformations
    # of the independent or control variables
    transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
    # for each variable in response_vars, check if it is transformed
    # in the transform_code by using an LLM assistant
    llm_assistant = llm(provider=llm_provider, model=llm_model)
    # for dict_idx, var in enumerate(response_vars):
    transform_responses = get_feature_transforms(llm_assistant,
                                                 transform_code,
                                                 response_vars['columns'],
                                                 response_vars['description'])
    response_vars['transform_code'] = [response.text[0].content \
        for response in transform_responses]

    # save updated response variables in features dict
    features[i]['response_variables'] = response_vars

[2025-11-12 09:03:00.47][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-12 09:03:01.16][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [11]:
# view feature dictionary to ensure correctness
features

{0: {'independent_variables': [{'description': 'Name femininity rating (continuous). This is the mean masculinity-femininity rating where larger values indicate more feminine names. Standardized for interpretability and to aid model convergence.',
    'columns': ['masfem_std'],
    'transform_code': ["df['masfem_std'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary gender label for the hurricane name (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["# Ensure gender_mf is binary numeric (0/1)\ndf['gender_mf'] = pd.to_numeric(df['gender_mf'], errors='coerce')"]}],
  'control_variables': [{'description': 'Maximum wind speed at landfall (measures storm strength/severity).',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['wind'],
    'transform_code': ["df['wind'] = pd.to_numeric(df['win

## Extract Model Class Used

In [12]:
# create dict to store model information
model_info = {}

In [13]:
# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):

    # get model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']

    # use helper function to get model information
    model_info[i] = get_model_information(llm_assistant, model_code)

In [14]:
# view model_info to ensure correctness
model_info

{0: '{\n  "model_library": "statsmodels (imported as statsmodels.api as sm)",\n  "model_class": "sm.GLM with family=sm.families.NegativeBinomial (fallback: sm.GLM with family=sm.families.Poisson)), and sm.OLS for the log-damage robustness check",\n  "model_parameters": "GLM NegativeBinomial: family=sm.families.NegativeBinomial() (uses default log link); fallback GLM Poisson: family=sm.families.Poisson() with fit(cov_type=\'HC0\') for robust SEs; OLS: sm.OLS(...).fit(cov_type=\'HC1\') for robust SEs. Design matrix: predictors = [\'masfem_std\',\'gender_mf\',\'wind\',\'category\',\'min\',\'year\',\'elapsedyrs\',\'masfem_x_category\']; constant added via sm.add_constant(X_model). Interaction created as masfem_x_category = masfem_std * (category - category.mean()).",\n  "model_formula_fitting_code": "try:\\n    nb_model = sm.GLM(y_counts, X_model, family=sm.families.NegativeBinomial()).fit()\\n    results[\'neg_bin_alldeaths\'] = nb_model\\nexcept Exception as e:\\n    pois = sm.GLM(y_coun

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [15]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [16]:
# run the transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    transformed_datasets[i] = transform_func(data.copy()) # use copy of dataset

# run the model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    model_results[i] = model_func(transformed_datasets[i].copy()) # use copy

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [17]:
# view the first model result as a sanity check
model_results[0]

{'neg_bin_alldeaths': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x709700b10e80>,
 'ols_log_ndam15': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x70976487f2e0>}

In [18]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-11-12 09:03:13.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-12 09:03:58.08][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  44.75 seconds
[2025-11-12 09:03:58.09][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-12 09:03:58.10][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-12 09:04:30.18][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  32.08 seconds
[2025-11-12 09:04:30.19][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [19]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts statistics related to the name femininity effect (masfem_std) from two fitted models\n    contained in model_output:\n      - \'neg_bin_alldeaths\' : GLM (Negative Binomial or fallback Poisson) predicting alldeaths\n      - \'ols_log_ndam15\'    : OLS predicting log_ndam15 (robust SEs)\n    \n    Returns a dict with keys:\n      - "object": a nested dict with extracted numeric results for masfem_std (and masfem_x_category),\n                  including coefficient, SE, p-value, 95% CI, and exponentiated effect (IRR or multiplicative change).\n      - "description": brief interpretation about whether the results support the hypothesis that\n                       more-feminine names are associated with outcomes consistent with fewer precautions\n                       (i.e., higher deaths / greater damage).\n    """\n    import numpy as np\n\n    out = {}\n    results = {}\n\n    def safe_extract(res, var):\n        """

In [20]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [21]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    interpretation_code = final_answer_code[i]
    
    # get the interpretation output
    interpretation_output = final_answers[i]
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-11-12 09:04:30.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-12 09:04:35.63][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.30 seconds
[2025-11-12 09:04:35.64][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-12 09:04:35.65][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-12 09:04:40.51][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.85 seconds
[2025-11-12 09:04:40.51][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [22]:
conclusions

{0: '{\n  "answer": "No",\n  "justification": "The primary negative-binomial model on deaths shows a positive but highly non‑significant effect of name femininity (coef=0.221, p=0.555; IRR=1.25, 95% CI 0.60–2.60). The OLS robustness check is in the opposite direction and also non‑significant. Overall there is no statistical support for the hypothesis."\n}',
 1: '{\n  "answer": "No",\n  "justification": "The estimates on name femininity (continuous and binary) are positive but not statistically significant (e.g., masfem_z coef = 0.029, p = 0.804; female_name coef = 0.120, p = 0.618), with wide CIs spanning both negative and positive effects. Therefore the analysis provides no evidence that more feminine names lead to higher fatalities (fewer precautions)."\n}'}